# Enhanced Index Tracking with MIQP

This notebook implements a Mixed-Integer Quadratic Programming (MIQP)
model for enhanced index tracking of the Swiss Market Index (SMI).

The model minimises tracking error variance while requiring a minimum
expected excess return relative to the SMI.

In [1]:
#Import
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB, quicksum

In [2]:
#Import Data, drop the columns Date and SMI Index
df=pd.read_csv("data/initialisation/prices_smi_20240315.csv", sep=";", header=0)
df=df.drop(labels=["Date","SMI Index"],axis=1)

# Define our T
T=df.shape[0]-1

In [3]:
#Calculate the returns
returns= df.select_dtypes(include=['number']).pct_change()

# remove first row with NaN values
returns=returns.iloc[1:]

In [4]:
#Calculate the covariance matrix
sigma=returns.cov()

In [5]:
#Create an empty model

model = gp.Model('portfolio')
model.setParam("FeasibilityTol", 1e-06)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-10-03


In [6]:
# U: complete SMI stock universe
U = df.columns.values

# I: stocks considered for portfolio construction
# All SMI constituents are included in this implementation.
I = U

In [7]:
#Read in the weights
weights = pd.read_csv("data/initialisation/close_smi_20240315.csv", sep=";", header=0, index_col=[0])

# Keep only the columns with names "Instrument" or "Weight"
selected_columns = weights.loc[:, weights.columns.isin(["Instrument_Name", "Weight"])]

#Reformate all the stock names to the "normal" form instead of all upper-case
# Custom function to format the names
def format_instrument_name(name):
    words = name.split()
    formatted_words = []
    for word in words:
        if word.upper() in ['ABB', 'UBS', 'AG', 'SA']:
            formatted_words.append(word.upper())
        else:
            formatted_words.append(word.capitalize())
    return ' '.join(formatted_words)

# Apply the custom function to the "Instrument_Name" column
selected_columns.loc[:, 'Instrument_Name'] = selected_columns['Instrument_Name'].apply(format_instrument_name)

#Set Instrument_Name as Index
selected_columns.set_index("Instrument_Name",inplace=True)
weights=selected_columns
#Transpose the Matrix
weights=weights.transpose()
# Convert weights to a dictionary 
weight_records = weights.to_dict("records")
w = weight_records[0]
# parse weights in the dictionary to floats and divide by 100 to get percent values
for stock, weight in w.items():
    w[stock] = float(weight) / 100
w

{'Nestle SA': 0.16321149999999998,
 'Novartis AG': 0.1532362,
 'Roche Holding AG': 0.1372492,
 'UBS Group AG': 0.07295829999999999,
 'Cie Financiere Richemont SA': 0.0663424,
 'Zurich Insurance Group AG': 0.0602797,
 'ABB Ltd': 0.0580187,
 'Sika AG': 0.035792199999999996,
 'Holcim AG': 0.034883700000000004,
 'Alcon Inc': 0.031941899999999995,
 'Lonza Group AG': 0.0298773,
 'Swiss Re AG': 0.027961399999999997,
 'Givaudan SA': 0.0270437,
 'Partners Group Holding AG': 0.024689100000000002,
 'Swiss Life Holding AG': 0.0158703,
 'Geberit AG': 0.0149628,
 'Logitech International SA': 0.011714800000000001,
 'Swisscom AG': 0.011364099999999999,
 'Kuehne + Nagel International AG': 0.011319,
 'Sonova Holding AG': 0.0112836}

In [8]:
# Model parameters

C = 100000       # Initial investment budget in CHF
alpha = 0.02     # Minimum expected annual excess return
epsilon = 0.01   # Minimum portfolio weight
delta = 0.30     # Maximum portfolio weight
k = 10           # Number of selected stocks

In [9]:
# Continuous variables: number of shares held
x = model.addVars(
    U,
    name="x",
    lb=0.0
)

# Binary variables: stock selection
z = model.addVars(
    U,
    name="z",
    vtype=GRB.BINARY
)

In [10]:
# Annualised historical mean returns
r_bar = returns.mean() * 240

## MIQP Formulation

The objective minimises the Tracking Error Variance (TEV) between the
portfolio and the SMI.

The model additionally imposes:

- a full-investment budget constraint,
- a minimum expected excess return,
- minimum and maximum portfolio weights,
- and a cardinality constraint.

In [11]:
#Objective Function
TEV = quicksum(
    sigma[i][j] * (((df[i][T] * x[i]) / C) - w[i]) * (((df[j][T] * x[j]) / C) - w[j])
    for i in I
    for j in I
) - 2 * quicksum(
    sigma[i][j] * (((df[i][T] * x[i]) / C) * w[i] - w[i] * w[j])
    for i in I
    for j in set(U) - set(I)
) + quicksum(
    sigma[j][j] * w[i] * w[j]
    for i in set(U) - set(I)
    for j in set(U) - set(I)
)

In [12]:
# Objective: minimise tracking error variance
model.setObjective(TEV * 1e6, GRB.MINIMIZE)


# Full investment
model.addConstr(quicksum(df[i][T]*x[i] for i in I)==C, 'budget constraint')

# Minimum expected excess return
model.addConstr(quicksum(((df[i][T]*x[i])/C)*r_bar[i] for i in I) - quicksum(w[i]*r_bar[i] for i in U)>=alpha, 'excess return')

# Minimum and maximum portfolio weights
model.addConstrs((epsilon*z[i] <=df[i][T]*x[i]/C for i in I), "min investment")
model.addConstrs((delta*z[i] >=df[i][T]*x[i]/C for i in I), "max investment")


# Exactly 10 stocks are selected
model.addConstr(
    quicksum(z[i] for i in I) == k,
    "cardinality_constraint"
)

# Verify model formulation
model.write('portfolio_selection_optimization.lp')

# Solve
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 43 rows, 40 columns and 140 nonzeros
Model fingerprint: 0x45fc73bd
Model has 210 quadratic objective terms
Variable types: 20 continuous, 20 integer (20 binary)
Coefficient statistics:
  Matrix range     [5e-06, 4e+03]
  Objective range  [6e-02, 5e+00]
  QObjective range [4e-07, 7e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [9e-02, 1e+05]
Presolve time: 0.00s
Presolved: 43 rows, 40 columns, 140 nonzeros
Presolved model has 210 quadratic objective terms
Variable types: 20 continuous, 20 integer (20 binary)
Found heuristic solution: objective 16.4621245
Found heuristic solution: objective 5.6059198

Root relaxation: objective 9.285544e-01, 83 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Un

In [13]:
# Extract optimised portfolio
result = pd.DataFrame(index=I)

for i in I:
    result.loc[i, "Weight"] = (
        x[i].X * df[i][T] / C * 100
    )
    result.loc[i, "Shares"] = x[i].X

result

,Weight,Shares
ABB Ltd,12.012412,283.311605
Cie Financiere Richemont SA,9.190212,63.033004
Geberit AG,0.000000,0.000000
Givaudan SA,6.737482,1.677242
Kuehne + Nagel International AG,0.000000,0.000000
Holcim AG,0.000000,0.000000
Lonza Group AG,3.600950,7.596941
Nestle SA,14.603874,153.353707
Novartis AG,17.070459,199.328107
Roche Holding AG,10.342165,44.810074


In [14]:
# Check portfolio constraints
print(f"Total portfolio weight: {result['Weight'].sum():.2f}%")
print(f"Selected stocks: {(result['Weight'] > 0).sum()}")

Total portfolio weight: 100.00%
Selected stocks: 10
